# Albert Heijn API notebook

Interactieve client voor de **officiële Albert Heijn mobile-app API** (`api.ah.nl`) — dezelfde API die de Appie-app gebruikt. Dit is de meest betrouwbare route die er is: geen scraping, geen third-party proxy, gewoon de echte backend.

De API bestaat uit twee delen:

| Onderdeel | Endpoint | Auth | Gebruikt voor |
|---|---|---|---|
| REST *mobile-services* | `https://api.ah.nl/mobile-services/...` | anoniem token volstaat voor producten | productzoeken, boodschappenlijstje |
| GraphQL | `https://api.ah.nl/graphql` | anoniem voor recepten, **lid-token** voor mandje | recepten, winkelmandje (basket) |

**Authenticatie** loopt via `https://api.ah.nl/mobile-auth/v1/auth/...` met `clientId: "appie"`:
- *Anoniem token* — één POST, direct klaar. Genoeg voor producten en recepten.
- *Lid-token* — OAuth-code flow via `login.ah.nl` (eenmalig code plakken). Nodig voor je winkelmandje. Tokens worden lokaal opgeslagen (`ah_tokens.json`, staat in `.gitignore`) en automatisch ge-refresht.

> ⚠️ **Disclaimer**: dit is een *ongedocumenteerde* API, reverse-engineered door de community. AH kan hem zonder aankondiging wijzigen (GraphQL-introspectie is begin 2026 al uitgezet — de queries hieronder komen uit een eerder gedumpt schema en werken nog). Gebruik met mate; hamer niet op de endpoints.

Bronnen: [jabbink's gist](https://gist.github.com/jabbink/8bfa44bdfc535d696b340c46d228fdd1) (jouw voorbeeld), [appie-go](https://github.com/gwillem/appie-go), [albert-heijn-graphql-api](https://github.com/JaapWestera/albert-heijn-graphql-api) (GraphQL-schema).


In [1]:
import json
import time
import webbrowser
from pathlib import Path

import requests
import pandas as pd

API_BASE = "https://api.ah.nl"
GRAPHQL_URL = f"{API_BASE}/graphql"
AUTH_BASE = f"{API_BASE}/mobile-auth/v1/auth"
TOKEN_FILE = Path("ah_tokens.json")

# De headers waarmee de Appie-app zich meldt. X-Application is vereist voor
# een aantal endpoints (o.a. boodschappenlijstje) en kan nergens kwaad.
BASE_HEADERS = {
    "User-Agent": "Appie/8.22.3",
    "Content-Type": "application/json",
    "X-Application": "AHWEBSHOP",
}

LOGIN_URL = (
    "https://login.ah.nl/secure/oauth/authorize"
    "?client_id=appie&redirect_uri=appie://login-exit&response_type=code"
)


## Client

`AHClient` regelt alle authenticatie:

- start automatisch met een **anoniem token** (producten + recepten werken meteen);
- `login()` opent de AH-loginpagina voor de **lid-flow** (winkelmandje);
- tokens worden bewaard in `ah_tokens.json` en bij verloop automatisch ge-refresht;
- bij een 401 wordt één keer ge-refresht en opnieuw geprobeerd.


In [2]:
class AHError(RuntimeError):
    pass


class AHClient:
    def __init__(self, token_file: Path = TOKEN_FILE):
        self.token_file = Path(token_file)
        self.session = requests.Session()
        self.session.headers.update(BASE_HEADERS)
        self.access_token = None
        self.refresh_token = None
        self.expires_at = 0.0
        self.is_member = False
        if not self._load_tokens():
            self.anonymous_login()

    # ---------- token opslag ----------

    def _save_tokens(self):
        self.token_file.write_text(json.dumps({
            "access_token": self.access_token,
            "refresh_token": self.refresh_token,
            "expires_at": self.expires_at,
            "is_member": self.is_member,
        }, indent=2))

    def _load_tokens(self) -> bool:
        if not self.token_file.exists():
            return False
        data = json.loads(self.token_file.read_text())
        self.access_token = data.get("access_token")
        self.refresh_token = data.get("refresh_token")
        self.expires_at = data.get("expires_at", 0)
        self.is_member = data.get("is_member", False)
        if time.time() > self.expires_at - 60:
            try:
                self.refresh()
            except AHError:
                if self.is_member:
                    print("⚠️ Refresh mislukt — log opnieuw in met ah.login()")
                self.anonymous_login()
        return self.access_token is not None

    def _store_token_response(self, resp: requests.Response, is_member: bool):
        if not resp.ok:
            raise AHError(f"Auth mislukt ({resp.status_code}): {resp.text[:300]}")
        data = resp.json()
        self.access_token = data["access_token"]
        self.refresh_token = data.get("refresh_token")
        self.expires_at = time.time() + data.get("expires_in", 7199)
        self.is_member = is_member
        self._save_tokens()

    # ---------- auth flows ----------

    def anonymous_login(self):
        resp = self.session.post(f"{AUTH_BASE}/token/anonymous",
                                 json={"clientId": "appie"})
        self._store_token_response(resp, is_member=False)
        print("✅ Anoniem token opgehaald (producten & recepten werken, mandje niet)")

    def login(self):
        """Open de AH-loginpagina en leg uit hoe je de code pakt."""
        print("1. Er opent zo een browservenster met de AH-login.")
        print("2. Log in. De browser probeert daarna naar appie://login-exit?code=... te gaan")
        print("   en toont een foutmelding — dat hoort zo.")
        print("3. Kopieer de waarde van 'code=' uit de adresbalk (of uit het")
        print("   DevTools Network-tabblad) en run: ah.login_with_code(\"<code>\")")
        webbrowser.open(LOGIN_URL)
        print(f"\nLukt het openen niet, ga zelf naar:\n{LOGIN_URL}")

    def login_with_code(self, code: str):
        resp = self.session.post(f"{AUTH_BASE}/token",
                                 json={"clientId": "appie", "code": code.strip()})
        self._store_token_response(resp, is_member=True)
        print("✅ Ingelogd als lid — winkelmandje beschikbaar")

    def refresh(self):
        if not self.refresh_token:
            raise AHError("Geen refresh token")
        resp = self.session.post(f"{AUTH_BASE}/token/refresh",
                                 json={"clientId": "appie",
                                       "refreshToken": self.refresh_token})
        self._store_token_response(resp, is_member=self.is_member)

    # ---------- request helpers ----------

    def _auth_headers(self):
        if time.time() > self.expires_at - 60:
            self.refresh()
        return {"Authorization": f"Bearer {self.access_token}"}

    def _request(self, method: str, url: str, **kwargs) -> requests.Response:
        resp = self.session.request(method, url, headers=self._auth_headers(), **kwargs)
        if resp.status_code == 401:  # token verlopen/ingetrokken: één keer opnieuw
            self.refresh()
            resp = self.session.request(method, url, headers=self._auth_headers(), **kwargs)
        if not resp.ok:
            raise AHError(f"{method} {url} -> {resp.status_code}: {resp.text[:300]}")
        return resp

    def rest(self, method: str, path: str, **kwargs):
        return self._request(method, f"{API_BASE}{path}", **kwargs).json()

    def graphql(self, query: str, variables: dict | None = None):
        resp = self._request("POST", GRAPHQL_URL,
                             json={"query": query, "variables": variables or {}})
        payload = resp.json()
        if payload.get("errors"):
            raise AHError(json.dumps(payload["errors"], indent=2)[:1000])
        return payload["data"]

    def require_member(self):
        if not self.is_member:
            raise AHError("Hiervoor moet je ingelogd zijn: run ah.login() "
                          "en daarna ah.login_with_code(code)")


ah = AHClient()


✅ Anoniem token opgehaald (producten & recepten werken, mandje niet)


## Inloggen als lid *(alleen nodig voor het winkelmandje)*

Run de cel hieronder, log in via de browser, en plak de `code` uit de mislukte
`appie://login-exit?code=...`-redirect in `login_with_code`. De refresh-token
blijft daarna lang geldig, dus dit hoef je zelden opnieuw te doen.


In [ ]:
#ah.login()                          # opent de browser met de AH-loginpagina
ah.login_with_code("0b7c1f5e-cda6-4a0a-8b71-060497c12f58")  # plak de code uit de redirect-URL

print(f"Ingelogd als lid: {ah.is_member}")


1. Er opent zo een browservenster met de AH-login.
2. Log in. De browser probeert daarna naar appie://login-exit?code=... te gaan
   en toont een foutmelding — dat hoort zo.
3. Kopieer de waarde van 'code=' uit de adresbalk (of uit het
   DevTools Network-tabblad) en run: ah.login_with_code("<code>")

Lukt het openen niet, ga zelf naar:
https://login.ah.nl/secure/oauth/authorize?client_id=appie&redirect_uri=appie://login-exit&response_type=code
✅ Ingelogd als lid — winkelmandje beschikbaar
Ingelogd als lid: True


## Producten zoeken *(werkt anoniem)*

REST-endpoint `GET /mobile-services/product/search/v2`. Dit heb je ook nodig om
`productId`s te vinden voor het mandje en het boodschappenlijstje.


In [7]:
def search_products(query: str, size: int = 10) -> pd.DataFrame:
    data = ah.rest("GET", "/mobile-services/product/search/v2",
                   params={"query": query, "sortOn": "RELEVANCE", "size": size})
    rows = [{
        "productId": p["webshopId"],
        "titel": p.get("title"),
        "merk": p.get("brand"),
        "prijs": p.get("priceBeforeBonus"),
        "bonus": p.get("isBonus", False),
        "eenheid": p.get("salesUnitSize"),
    } for p in data.get("products", [])]
    return pd.DataFrame(rows)


search_products("kaas")


,productId,titel,merk,prijs,bonus,eenheid
0,572638,Milner Jong belegen 35+ plakken,Milner,2.99,False,150 g
1,572641,Milner Jong 35+ plakken voordeelpak,Milner,4.39,False,250 g
2,197226,De Zaanse Hoeve Goudse jong 48+ plakken voordeel,De Zaanse Hoeve,3.95,False,400 g
3,197227,De Zaanse Hoeve Jong belegen 48+ plakken voord...,De Zaanse Hoeve,4.15,False,400 g
4,602967,AH Goudse Jong belegen 48+ stuk kvp 2-pack,AH,14.72,True,2 stuks
5,421292,Milner Oud 30+ plakken,Milner,3.69,False,150 g
6,618102,AH Zaanlander Jong belegen 48+ stuk 3-pack,AH Zaanlander,23.67,True,3 stuks
7,618879,AH Goudse jong belegen 48+ stuk 3-pack,AH,22.08,True,3 stuks
8,447741,De Zaanse Hoeve Goudse jong belegen 48+ stuk v...,De Zaanse Hoeve,8.80,False,ca. 1006 g
9,2600,AH Goudse jong 48+ plakken,AH,2.95,True,190 g


## Recepten *(werkt anoniem)*

Recepten (Allerhande) zitten volledig in GraphQL: `recipeSearch` voor zoeken,
`recipe(id:)` voor het volledige recept met ingrediënten en bereidingsstappen.
Sorteren kan op `MOST_RELEVANT`, `NEWEST`, `POPULAR`, `RATINGS`, `TOTAL_TIME` of `TRENDING`.


In [10]:
RECIPE_SEARCH_QUERY = """
query SearchRecipes($searchText: String, $start: Int, $size: PageSize,
                    $sortBy: RecipeSearchSortOption) {
  recipeSearch(query: {searchText: $searchText, start: $start,
                       size: $size, sortBy: $sortBy}) {
    page { total hasNextPage }
    result {
      id
      title
      slug
      courses
      diet
      nutriScore
      time { cook oven wait }
      rating { average count }
      serving { number type }
      images(renditions: [D440X324]) { url }
    }
  }
}
"""


def search_recipes(text: str, size: int = 10, start: int = 0,
                   sort_by: str = "MOST_RELEVANT") -> pd.DataFrame:
    data = ah.graphql(RECIPE_SEARCH_QUERY, {
        "searchText": text, "start": start, "size": size, "sortBy": sort_by,
    })
    res = data["recipeSearch"]
    print(f"{res['page']['total']} recepten gevonden voor '{text}'")
    rows = [{
        "recipeId": r["id"],
        "titel": r["title"],
        "gang": ", ".join(r["courses"]),
        "dieet": ", ".join(d for d in r["diet"] if d),
        "kooktijd (min)": r["time"]["cook"],
        "rating": r["rating"]["average"],
        "nutriScore": r["nutriScore"],
        "personen": r["serving"]["number"],
        "url": f"https://www.ah.nl/allerhande/recept/R-R{r['id']}/{r['slug']}",
    } for r in res["result"]]
    return pd.DataFrame(rows)


recepten = search_recipes("zalm")
recepten


972 recepten gevonden voor 'zalm'


,recipeId,titel,gang,dieet,kooktijd (min),rating,nutriScore,personen,url
0,1201316,Zalm tataki,voorgerecht,lactosevrij,15,5.0,None,4,https://www.ah.nl/allerhande/recept/R-R1201316...
1,1201317,Zalm sashimi,voorgerecht,lactosevrij,10,5.0,None,4,https://www.ah.nl/allerhande/recept/R-R1201317...
2,1197300,Zalm 'puttanesca',hoofdgerecht,,20,5.0,None,4,https://www.ah.nl/allerhande/recept/R-R1197300...
3,1202441,Pinsa met zalm,lunch,,10,NaN,None,2,https://www.ah.nl/allerhande/recept/R-R1202441...
4,861186,Pizza met zalm,hoofdgerecht,,10,3.0,None,2,https://www.ah.nl/allerhande/recept/R-R861186/...
5,775784,Blini's met zalm,borrelhapje,zonder vlees,10,4.0,None,16,https://www.ah.nl/allerhande/recept/R-R775784/...
6,1200565,Zalm-wraprolletjes,borrelhapje,,15,4.0,None,20,https://www.ah.nl/allerhande/recept/R-R1200565...
7,1201192,Noedels met zalm,hoofdgerecht,,20,4.0,None,2,https://www.ah.nl/allerhande/recept/R-R1201192...
8,1199777,Marinade voor zalm,,,5,4.0,None,4,https://www.ah.nl/allerhande/recept/R-R1199777...
9,1200575,Quiche spinazie zalm,hoofdgerecht,,25,4.0,None,4,https://www.ah.nl/allerhande/recept/R-R1200575...


In [11]:
RECIPE_DETAILS_QUERY = """
query GetRecipe($id: Int!, $servings: Int) {
  recipe(id: $id, servings: $servings) {
    id
    title
    description
    courses
    cuisines
    cookTime
    ovenTime
    waitTime
    nutriScore
    rating { average count }
    servings { number type min max isChangeable }
    ingredients {
      id
      quantity
      quantityUnit { singular plural }
      name { singular plural }
      text
    }
    preparation { steps summary }
  }
}
"""


def get_recipe(recipe_id: int, servings: int | None = None) -> dict:
    return ah.graphql(RECIPE_DETAILS_QUERY,
                      {"id": int(recipe_id), "servings": servings})["recipe"]


def print_recipe(recipe: dict):
    r = recipe
    tijd = f"{r['cookTime']} min koken"
    if r.get("ovenTime"):
        tijd += f" + {r['ovenTime']} min oven"
    if r.get("waitTime"):
        tijd += f" + {r['waitTime']} min wachten"
    print(f"🍽  {r['title']}  ({r['servings']['number']} {r['servings']['type']})")
    print(f"    {tijd} | NutriScore {r['nutriScore']} | "
          f"⭐ {r['rating']['average']} ({r['rating']['count']}x)\n")
    print("Ingrediënten:")
    for ing in r["ingredients"]:
        line = ing.get("text")
        if not line:
            qty = ing.get("quantity")
            unit = (ing.get("quantityUnit") or {}).get("singular") or ""
            naam = ing["name"]["singular"]
            line = " ".join(str(p) for p in (qty, unit, naam) if p)
        print(f"  - {line}")
    print("\nBereiding:")
    for i, step in enumerate(r["preparation"]["steps"], 1):
        print(f"  {i}. {step}")


# Pak het eerste zoekresultaat van hierboven
recept = get_recipe(recepten.iloc[0]["recipeId"])
print_recipe(recept)


🍽  Zalm tataki  (4 porties)
    15 min koken + 10 min wachten | NutriScore None | ⭐ 4.875001 (24x)

Ingrediënten:
  - 5 el sojasaus
  - 2 el rijstazijn
  - 1 el sesamolie
  - 1 el gembersiroop
  - 300 g sushizalm
  - 5 el sesamzaad
  - 125 g zeewiersalade
  - 5 g pittige kiemgroente

Bereiding:
  1. Meng de sojasaus met de rijstazijn, sesamolie en de gembersiroop tot een gladde marinade. Doe ⅔ van de marinade in een diep bord.
  2. Dep de stukken zalm droog met keukenpapier en wentel door het sojasausmengsel, laat 10 min. marineren.
  3. Verdeel het sesamzaad over een plat bord en rol de gemarineerde zalm erdoor, zodat deze aan alle kanten goed bedekt is. 
  4. Verhit de rest van de sesamolie in een koekenpan op hoog vuur.
  5.  Zodra de olie goed warm is, schroei je de zalm aan alle kanten dicht, ca. 10 sec. per kant. Haal de zalm direct uit de pan en laat 2 min. rusten. 
  6. Snijd de zalm met een scherp mes in plakken van 0.5 cm.
  7. Verdeel de zeewiersalade over een bord en leg de

## Winkelmandje *(lid-login vereist)*

Het échte online-bestelmandje (`basket`) zit in GraphQL. Ophalen met de
`basket`-query; muteren met `basketItemsAdd`, `basketItemsUpdate` en
`basketItemsDelete`. Items voeg je toe op `productId` (het `webshopId` uit
`search_products`).


In [12]:
BASKET_QUERY = """
query GetBasket {
  basket {
    summary {
      quantity
      price {
        totalPrice { amount formatted }
        discount { amount formatted }
      }
    }
    items { id quantity description }
    products {
      id
      quantity
      product {
        id
        title
        brand
        price { now { amount formatted } }
      }
    }
  }
}
"""


def get_basket() -> pd.DataFrame:
    ah.require_member()
    basket = ah.graphql(BASKET_QUERY)["basket"]
    s = basket["summary"]
    print(f"🛒 {s['quantity']} artikelen | totaal {s['price']['totalPrice']['formatted']}")
    rows = [{
        "productId": p["id"],
        "titel": p["product"]["title"],
        "merk": p["product"]["brand"],
        "aantal": p["quantity"],
        "prijs": p["product"]["price"]["now"]["formatted"],
    } for p in basket["products"]]
    return pd.DataFrame(rows)


get_basket()


AHError: POST https://api.ah.nl/graphql -> 400: {"errors":[{"message":"Cannot query field \"description\" on type \"BasketItem\".","locations":[{"line":11,"column":25}],"extensions":{"code":"GRAPHQL_VALIDATION_FAILED"}}]}

In [ ]:
ADD_TO_BASKET = """
mutation AddToBasket($items: [BasketMutation!]!) {
  basketItemsAdd(items: $items) { status errorMessage }
}
"""

UPDATE_BASKET = """
mutation UpdateBasket($items: [BasketMutation!]!) {
  basketItemsUpdate(items: $items) { status errorMessage }
}
"""

DELETE_FROM_BASKET = """
mutation DeleteFromBasket($items: [BasketDelete!]!) {
  basketItemsDelete(items: $items) { status errorMessage }
}
"""


def _check(result: dict):
    if result["status"] != "SUCCESS":
        raise AHError(f"{result['status']}: {result.get('errorMessage')}")
    print(f"✅ {result['status']}")


def add_to_basket(product_id: int, quantity: int = 1):
    """Voeg een product toe aan het mandje (product_id = webshopId)."""
    ah.require_member()
    data = ah.graphql(ADD_TO_BASKET,
                      {"items": [{"id": int(product_id), "quantity": quantity}]})
    _check(data["basketItemsAdd"])


def set_basket_quantity(product_id: int, quantity: int):
    ah.require_member()
    data = ah.graphql(UPDATE_BASKET,
                      {"items": [{"id": int(product_id), "quantity": quantity}]})
    _check(data["basketItemsUpdate"])


def remove_from_basket(product_id: int):
    ah.require_member()
    data = ah.graphql(DELETE_FROM_BASKET, {"items": [{"id": int(product_id)}]})
    _check(data["basketItemsDelete"])


# Voorbeeld: zoek een product en leg het in het mandje
hits = search_products("AH pindakaas")
display(hits.head(3))

# add_to_basket(hits.iloc[0]["productId"])   # ontcommentarieer om echt toe te voegen
# get_basket()


## Boodschappenlijstje *(lid-login vereist)*

Naast het bestelmandje heeft AH ook het *boodschappenlijstje* uit de app.
Dat loopt via REST: `GET /mobile-services/lists/v3/lists` en
`PATCH /mobile-services/shoppinglist/v2/items`.


In [ ]:
def get_shopping_lists():
    ah.require_member()
    return ah.rest("GET", "/mobile-services/lists/v3/lists")


def add_to_shopping_list(product_id: int, quantity: int = 1):
    """Zet een product op het boodschappenlijstje (niet het bestelmandje)."""
    ah.require_member()
    return ah.rest("PATCH", "/mobile-services/shoppinglist/v2/items",
                   json={"items": [{
                       "originCode": "PRD",
                       "productId": int(product_id),
                       "quantity": quantity,
                       "type": "SHOPPABLE",
                   }]})


# get_shopping_lists()
# add_to_shopping_list(58444)  # voorbeeld-productId


## Van recept naar mandje 🍝→🛒

De hele pijplijn aan elkaar geknoopt: zoek per ingrediënt van een recept het
beste productmatch en leg alles in het mandje. Standaard `dry_run=True`, zodat
je eerst ziet wat er toegevoegd zou worden.


In [ ]:
def recipe_to_basket(recipe_id: int, dry_run: bool = True) -> pd.DataFrame:
    recipe = get_recipe(recipe_id)
    print(f"Ingrediënten voor: {recipe['title']}\n")
    rows = []
    for ing in recipe["ingredients"]:
        naam = ing["name"]["singular"]
        hits = search_products(naam, size=1)
        if hits.empty:
            rows.append({"ingrediënt": naam, "product": "⚠️ geen match",
                         "productId": None, "prijs": None})
            continue
        p = hits.iloc[0]
        rows.append({"ingrediënt": naam, "product": p["titel"],
                     "productId": p["productId"], "prijs": p["prijs"]})
    plan = pd.DataFrame(rows)

    if dry_run:
        print("Dry-run: nog niets toegevoegd. Run met dry_run=False om het mandje te vullen.")
    else:
        ah.require_member()
        for pid in plan["productId"].dropna():
            add_to_basket(int(pid))
        print("Klaar — check je mandje met get_basket()")
    return plan


recipe_to_basket(recepten.iloc[0]["recipeId"], dry_run=True)


---
### Tips & referentie

- **Rate limits**: de API is niet publiek gedocumenteerd; houd het bij enkele requests per seconde.
- **Token kwijt/geweigerd?** Verwijder `ah_tokens.json` en run de client-cel opnieuw (anoniem), of doorloop `ah.login()` opnieuw voor lid-toegang.
- **Meer GraphQL**: het schema kent nog veel meer, o.a. `recipeSearchV2` (met filters per groep), `recipeSuggestionsForProductId`, `recipesRelatedToRecipe`, `scrapeRecipe(url:)` (importeert een recept van een externe site!), `basketMergeList` (lijstje → mandje) en `receipts` (kassabonnen).
- **België**: gebruik `login.ah.be` in plaats van `login.ah.nl`; de API-host blijft `api.ah.nl`.
